In [18]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
import re

In [4]:
df = pd.read_csv("../../train_corrigido.csv", sep=";",encoding="utf-8-sig")
df = df.fillna("")
df.head(2)

,medical_specialty,description,sample_name,transcription,keywords
0,Cardiovascular-Pulmonary,"Chest pain, possible syncopal spells. She has...",Chest Pain - Cardiac Consult,"REASON FOR REFERRAL: Chest pain, possible syn...","cardiovascular / pulmonary, chest pain, syncop..."
1,Cardiovascular-Pulmonary,"The patient is 14 months old, comes in with a ...",Difficulty Breathing - ER Visit,"HISTORY: The patient is 14 months old, comes ...",


In [5]:
X = df["transcription"]
y = df["medical_specialty"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)


In [6]:
model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
            max_features=50_000,
        ),
    ),
    (
        "classifier",
        LinearSVC(
            C=0.5,
            class_weight="balanced",
        ),
    ),
])

In [7]:
model.fit(X_train, y_train)

predictions = model.predict(X_valid)

In [8]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

accuracy = accuracy_score(y_valid, predictions)
macro_f1 = f1_score(
    y_valid,
    predictions,
    average="macro",
)

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro-F1: {macro_f1:.4f}")

Accuracy: 0.3565
Macro-F1: 0.4095


In [9]:
print(
    classification_report(
        y_valid,
        predictions,
        zero_division=0,
    )
)

                          precision    recall  f1-score   support

Cardiovascular-Pulmonary       0.33      0.32      0.33        62
             Dermatology       1.00      0.60      0.75         5
        Gastroenterology       0.23      0.23      0.23        39
        General Medicine       0.58      0.90      0.70        42
               Neurology       0.33      0.32      0.33        37
            Neurosurgery       0.09      0.12      0.11        16
   Obstetrics-Gynecology       0.29      0.30      0.29        27
           Ophthalmology       0.50      0.80      0.62        15
              Orthopedic       0.30      0.33      0.31        60
   Psychiatry-Psychology       0.75      0.75      0.75         8
               Radiology       0.16      0.16      0.16        44
                 Surgery       0.39      0.30      0.34       178

                accuracy                           0.36       533
               macro avg       0.41      0.43      0.41       533
        

In [10]:
def combine_fields(data):
    return (
        data["description"].fillna("")
        + " "
        + data["sample_name"].fillna("")
        + " "
        + data["transcription"].fillna("")
    )

In [11]:
X_transcription = df["transcription"].fillna("")

X_text = (
    df["description"].fillna("")
    + " "
    + df["transcription"].fillna("")
)

X_all_without_keywords = (
    df["description"].fillna("")
    + " "
    + df["sample_name"].fillna("")
    + " "
    + df["transcription"].fillna("")
)

X_all_fields = (
    df["description"].fillna("")
    + " "
    + df["sample_name"].fillna("")
    + " "
    + df["transcription"].fillna("")
    + " "
    + df["keywords"].fillna("")
)

In [12]:
from sklearn.model_selection import train_test_split

train_indices, valid_indices = train_test_split(
    df.index,
    test_size=0.2,
    random_state=42,
    stratify=df["medical_specialty"],
)

y_train = df.loc[train_indices, "medical_specialty"]
y_valid = df.loc[valid_indices, "medical_specialty"]

In [13]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate_input(X, name):
    model = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=2,
                sublinear_tf=True,
                max_features=50_000,
            ),
        ),
        (
            "classifier",
            LinearSVC(
                C=0.5,
                class_weight="balanced",
            ),
        ),
    ])

    model.fit(
        X.loc[train_indices],
        y_train,
    )

    predictions = model.predict(X.loc[valid_indices])

    return {
        "input": name,
        "accuracy": accuracy_score(y_valid, predictions),
        "macro_f1": f1_score(
            y_valid,
            predictions,
            average="macro",
        ),
    }

In [14]:
results = pd.DataFrame([
    evaluate_input(
        X_transcription,
        "transcription",
    ),
    evaluate_input(
        X_text,
        "description + transcription",
    ),
    evaluate_input(
        X_all_without_keywords,
        "description + sample_name + transcription",
    ),
    evaluate_input(
        X_all_fields,
        "all fields including keywords",
    ),
])

results

,input,accuracy,macro_f1
0,transcription,0.356473,0.409525
1,description + transcription,0.365854,0.412590
2,description + sample_name + transcription,0.362101,0.410005
3,all fields including keywords,0.440901,0.491775


#### Aqui faço a analise das keywords e as funções que permitem comparar as keywords com as Medical Specialties.
 Verifica-se que todas linhas que têm keywords, a primeira keyword corresponde à Medical Specialty 

In [39]:

def first_keyword(value):
    value = str(value).strip()
    if not value:
        return ""
    return value.split(",")[0].strip()

def word_normalized(value):
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return " ".join(value.split())

In [41]:
labels = set(
    df["medical_specialty"]
    .str.strip()
    .str.lower()
)

df["first_keyword"] = df["keywords"].map(first_keyword)
df["normalized_label"] = df["medical_specialty"].map(word_normalized)
df["normalized_first_keyword"] = df["first_keyword"].map(word_normalized) 
df["keyword_reveals_label"] = (
    df["normalized_first_keyword"] == df["normalized_label"]
)


#### Logistic Regression


In [42]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

logistic_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
            max_features=50_000,
        ),
    ),
    (
        "classifier",
        LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=2000,
            random_state=42,
        ),
    ),
])

In [43]:
logistic_model.fit(
    X_all_fields.loc[train_indices],
    y_train,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](12,)","['Cardiovascular-Pulmonary','Dermatology','Gastroenterology',..., 'Psychiatry-Psychology','Radiology','Surgery']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",2
,"max_features max_features: int, default=NoneIf not None, build a vocabulary that only consider the top`max_features` ordered by term frequency across the corpus.Otherwise, all features are used.This parameter is ignored if vocabulary is not None.",50000
,"sublinear_tf sublinear_tf: bool, default=FalseApply sublinear tf scaling, i.e. replace tf with 1 + log(tf).",True
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'


In [44]:
logistic_predictions = logistic_model.predict(
    X_all_fields.loc[valid_indices]
)

In [45]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
)

logistic_accuracy = accuracy_score(
    y_valid,
    logistic_predictions,
)

logistic_macro_f1 = f1_score(
    y_valid,
    logistic_predictions,
    average="macro",
    zero_division=0,
)

print(f"Accuracy: {logistic_accuracy:.4f}")
print(f"Macro-F1: {logistic_macro_f1:.4f}")

print(
    classification_report(
        y_valid,
        logistic_predictions,
        zero_division=0,
    )
)

Accuracy: 0.5084
Macro-F1: 0.5531
                          precision    recall  f1-score   support

Cardiovascular-Pulmonary       0.51      0.60      0.55        62
             Dermatology       1.00      0.60      0.75         5
        Gastroenterology       0.45      0.59      0.51        39
        General Medicine       0.54      0.95      0.69        42
               Neurology       0.42      0.30      0.35        37
            Neurosurgery       0.29      0.50      0.36        16
   Obstetrics-Gynecology       0.50      0.74      0.60        27
           Ophthalmology       0.56      1.00      0.71        15
              Orthopedic       0.46      0.63      0.54        60
   Psychiatry-Psychology       0.78      0.88      0.82         8
               Radiology       0.33      0.32      0.33        44
                 Surgery       0.71      0.31      0.43       178

                accuracy                           0.51       533
               macro avg       0.55     

In [46]:
model_results = pd.DataFrame([
    {
        "model": "Linear SVM",
        "input": "all fields",
        "augmentation": "none",
        "accuracy": 0.440901,
        "macro_f1": 0.491775,
    },
    {
        "model": "Logistic Regression",
        "input": "all fields",
        "augmentation": "none",
        "accuracy": logistic_accuracy,
        "macro_f1": logistic_macro_f1,
    },
])

model_results

,model,input,augmentation,accuracy,macro_f1
0,Linear SVM,all fields,none,0.440901,0.491775
1,Logistic Regression,all fields,none,0.508443,0.553089


In [53]:
svm_accuracy = accuracy_score(
    y_valid,
    all_fields_predictions,
)

svm_macro_f1 = f1_score(
    y_valid,
    all_fields_predictions,
    average="macro",
    zero_division=0,
)

model_results = pd.DataFrame([
    {
        "model": "Linear SVM",
        "input": "all fields",
        "augmentation": "none",
        "accuracy": svm_accuracy,
        "macro_f1": svm_macro_f1,
    },
    {
        "model": "Logistic Regression",
        "input": "all fields",
        "augmentation": "none",
        "accuracy": logistic_accuracy,
        "macro_f1": logistic_macro_f1,
    },
])

display(model_results)

NameError: name 'all_fields_predictions' is not defined

In [49]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

naive_bayes_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
            max_features=50_000,
        ),
    ),
    (
        "classifier",
        MultinomialNB(),
    ),
])

naive_bayes_model.fit(
    X_all_fields.loc[train_indices],
    y_train,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[<U24](12,)","['Cardiovascular-Pulmonary','Dermatology','Gastroenterology',..., 'Psychiatry-Psychology','Radiology','Surgery']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",2
,"max_features max_features: int, default=NoneIf not None, build a vocabulary that only consider the top`max_features` ordered by term frequency across the corpus.Otherwise, all features are used.This parameter is ignored if vocabulary is not None.",50000
,"sublinear_tf sublinear_tf: bool, default=FalseApply sublinear tf scaling, i.e. replace tf with 1 + log(tf).",True
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'


In [50]:
naive_bayes_predictions = naive_bayes_model.predict(
    X_all_fields.loc[valid_indices]
)

In [51]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
)

naive_bayes_accuracy = accuracy_score(
    y_valid,
    naive_bayes_predictions,
)

naive_bayes_macro_f1 = f1_score(
    y_valid,
    naive_bayes_predictions,
    average="macro",
    zero_division=0,
)

print(f"Accuracy: {naive_bayes_accuracy:.4f}")
print(f"Macro-F1: {naive_bayes_macro_f1:.4f}")

print(
    classification_report(
        y_valid,
        naive_bayes_predictions,
        zero_division=0,
    )
)

Accuracy: 0.4409
Macro-F1: 0.1625
                          precision    recall  f1-score   support

Cardiovascular-Pulmonary       0.50      0.18      0.26        62
             Dermatology       0.00      0.00      0.00         5
        Gastroenterology       0.00      0.00      0.00        39
        General Medicine       0.46      0.79      0.58        42
               Neurology       0.53      0.27      0.36        37
            Neurosurgery       0.00      0.00      0.00        16
   Obstetrics-Gynecology       0.00      0.00      0.00        27
           Ophthalmology       0.00      0.00      0.00        15
              Orthopedic       0.17      0.02      0.03        60
   Psychiatry-Psychology       0.00      0.00      0.00         8
               Radiology       0.23      0.07      0.11        44
                 Surgery       0.44      0.99      0.61       178

                accuracy                           0.44       533
               macro avg       0.19     

In [52]:
model_results = pd.DataFrame([
    {
        "model": "Linear SVM",
        "input": "all fields",
        "augmentation": "none",
        "accuracy": svm_accuracy,
        "macro_f1": svm_macro_f1,
    },
    {
        "model": "Logistic Regression",
        "input": "all fields",
        "augmentation": "none",
        "accuracy": logistic_accuracy,
        "macro_f1": logistic_macro_f1,
    },
    {
        "model": "Multinomial Naive Bayes",
        "input": "all fields",
        "augmentation": "none",
        "accuracy": naive_bayes_accuracy,
        "macro_f1": naive_bayes_macro_f1,
    },
])

display(
    model_results.sort_values(
        "macro_f1",
        ascending=False,
    )
)

NameError: name 'svm_accuracy' is not defined